# 13 — Temporal Versioning (Milestone M5)

**DSML stage:** modeling (bitemporal layer). Notebook 12 loaded risk factors from **multiple annual
filings per company**; right now every risk edge is `Active`, so a risk from a 2023 10-K looks as current
as one from the 2026 10-K. This notebook operates the bitemporal pattern from the feasibility studies:

1. **Cluster** each company's risks across its annual filings by embedding similarity — "the same risk,
   re-disclosed each year" becomes one lineage
2. **Backdate** `start_date` to the lineage's first disclosure (the citable first event)
3. **Close** lineages absent from the company's latest annual: `status='Deleted'`, `end_date` = latest
   filing date — a deleted 2023 risk must never be presented as an active 2026 risk
4. **Time-travel demo**: risks active as-of any date; newly added vs dropped risks per company

All updates are idempotent — re-running recomputes the same states.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()

SAME_RISK_SIM = 0.75  # cosine similarity above which two summaries are the same recurring risk

def run_cypher(q, **params):
    with driver.session() as s:
        return [dict(r) for r in s.run(q, **params)]
print("ready")

In [ ]:
# --- 0. One-time data-quality fix: normalize RiskFactor categories (idempotent) ---
# Notebook 12's live run revealed the extractor ignored the category enum's casing and invented
# new ones: the graph holds 'Supply Chain' AND 'supply chain', 'Cybersecurity' AND 'cybersecurity'
# (not in the enum at all), etc. Case-sensitive category filters (here, nb10/11 retrieval, nb12
# stage 7) silently miss half the data. Map everything to one canonical spelling.
CANONICAL_CATEGORIES = ["Supply Chain", "Geopolitical", "Export Controls", "Demand", "Competition",
                        "Technology", "Legal/Regulatory", "Financial", "Cybersecurity", "Other"]
canon_by_lower = {c.lower(): c for c in CANONICAL_CATEGORIES}

distinct = [r["c"] for r in run_cypher("MATCH (rf:RiskFactor) RETURN DISTINCT rf.category AS c") if r["c"]]
mapping = [{"old": c, "new": canon_by_lower.get(c.strip().lower(), c.strip().title())}
           for c in distinct]
changes = [m for m in mapping if m["old"] != m["new"]]
with driver.session() as s:
    s.run("""UNWIND $rows AS row
        MATCH (rf:RiskFactor {category: row.old}) SET rf.category = row.new""", rows=changes)
    dist = s.run("MATCH (rf:RiskFactor) RETURN rf.category AS c, count(*) AS n ORDER BY n DESC").data()
print(f"normalized {len(changes)} category spellings -> {len({m['new'] for m in mapping})} canonical values")
print({r["c"]: r["n"] for r in dist[:10]})

rows = run_cypher("""
    MATCH (c:Company)-[:DISCLOSES_RISK]->(rf:RiskFactor)-[:HAS_EVIDENCE]->(e:EvidenceSpan)
          -[:FROM_SECTION]->(:FilingSection)<-[:HAS_SECTION]-(f:Filing)
    WHERE f.form IN ['10-K', '20-F'] AND rf.embedding IS NOT NULL
    RETURN DISTINCT c.cik AS cik, c.name AS company, rf.risk_id AS risk_id, rf.summary AS summary,
           rf.category AS category, rf.embedding AS embedding,
           toString(f.filing_date) AS filing_date, f.accession_no AS accession_no
""")
risks = pd.DataFrame(rows)
risks["filing_date"] = pd.to_datetime(risks["filing_date"])
timeline = (risks.groupby("company")
            .agg(annual_filings=("accession_no", "nunique"), risk_nodes=("risk_id", "count"))
            .sort_values("annual_filings", ascending=False))
print(f"{len(risks)} annual-filing risk disclosures across {risks['company'].nunique()} companies")
timeline

In [ ]:
rows = run_cypher("""
    MATCH (c:Company)-[:DISCLOSES_RISK]->(rf:RiskFactor)-[:HAS_EVIDENCE]->(e:EvidenceSpan)
          -[:FROM_SECTION]->(:FilingSection)<-[:HAS_SECTION]-(f:Filing)
    WHERE f.form IN ['10-K', '20-F']
    RETURN c.cik AS cik, c.name AS company, rf.risk_id AS risk_id, rf.summary AS summary,
           rf.category AS category, rf.embedding AS embedding,
           toString(f.filing_date) AS filing_date, f.accession_no AS accession_no
""")
risks = pd.DataFrame(rows)
risks["filing_date"] = pd.to_datetime(risks["filing_date"])
timeline = (risks.groupby("company")
            .agg(annual_filings=("accession_no", "nunique"), risk_nodes=("risk_id", "count"))
            .sort_values("annual_filings", ascending=False))
print(f"{len(risks)} annual-filing risk disclosures across {risks['company'].nunique()} companies")
timeline

## 2. Cluster into lineages and compute temporal states

Greedy clustering per company (risks sorted by date; each joins the most similar existing lineage above
the threshold, else starts a new one). Simple, deterministic, and easy to audit — swap in something
fancier only if M6 evaluation shows it failing.

In [ ]:
updates = []  # one row per risk node: first_seen, last_seen, status, end_date, lineage_id
lineage_stats = []
for (cik, company), grp in risks.groupby(["cik", "company"]):
    grp = grp.sort_values("filing_date")
    latest_annual = grp["filing_date"].max()
    n_annuals = grp["accession_no"].nunique()
    embs = np.vstack(grp["embedding"].to_numpy())
    lineages: list[dict] = []  # {centroid, members(idx), dates}
    for i, (_, r) in enumerate(grp.iterrows()):
        best, best_sim = None, 0.0
        for lin in lineages:
            sim = float(embs[i] @ lin["centroid"])
            if sim > best_sim:
                best, best_sim = lin, sim
        if best is not None and best_sim >= SAME_RISK_SIM:
            best["members"].append(i)
            member_vecs = embs[best["members"]]
            centroid = member_vecs.mean(axis=0)
            best["centroid"] = centroid / np.linalg.norm(centroid)
        else:
            lineages.append({"centroid": embs[i], "members": [i]})
    for li, lin in enumerate(lineages):
        members = grp.iloc[lin["members"]]
        first_seen, last_seen = members["filing_date"].min(), members["filing_date"].max()
        # a lineage is closed when the company's latest annual no longer discloses it
        closed = (last_seen < latest_annual) and (n_annuals > 1)
        for _, m in members.iterrows():
            updates.append({"risk_id": m["risk_id"], "lineage_id": f"{cik}:{li}",
                            "first_seen": str(first_seen.date()), "last_seen": str(last_seen.date()),
                            "status": "Deleted" if closed else "Active",
                            "end_date": str(latest_annual.date()) if closed else None})
    n_closed = sum(1 for lin in lineages
                   if grp.iloc[lin['members']]["filing_date"].max() < latest_annual and n_annuals > 1)
    lineage_stats.append({"company": company, "annuals": n_annuals, "risk_nodes": len(grp),
                          "lineages": len(lineages), "closed_lineages": n_closed})
updates_df = pd.DataFrame(updates)
pd.DataFrame(lineage_stats).sort_values("closed_lineages", ascending=False)

## 3. Write temporal state back to the graph (idempotent)

In [ ]:
with driver.session() as s:
    s.run("""UNWIND $rows AS row
        MATCH (:Company)-[d:DISCLOSES_RISK]->(rf:RiskFactor {risk_id: row.risk_id})
        SET rf.lineage_id = row.lineage_id, rf.first_seen = date(row.first_seen),
            rf.last_seen = date(row.last_seen),
            d.start_date = date(row.first_seen), d.status = row.status,
            d.end_date = CASE WHEN row.end_date IS NULL THEN null ELSE date(row.end_date) END""",
          rows=updates_df.to_dict("records"))
    counts = s.run("""MATCH ()-[d:DISCLOSES_RISK]->() RETURN d.status AS status, count(*) AS n""").data()
print({c['status']: c['n'] for c in counts})

## 4. Time-travel demos — what the bitemporal pattern buys us

In [ ]:
# A. Risks Nvidia disclosed in its FY2023 10-K era that are GONE from the latest 10-K
dropped = run_cypher("""
    MATCH (c:Company {ticker:'NVDA'})-[d:DISCLOSES_RISK {status:'Deleted'}]->(rf:RiskFactor)
    RETURN DISTINCT rf.lineage_id AS lineage, rf.category AS category,
           left(rf.summary, 90) AS summary, toString(d.start_date) AS since, toString(d.end_date) AS closed
    ORDER BY closed DESC LIMIT 8""")
print(f"Nvidia risks dropped from the latest 10-K: {len(dropped)} lineages (showing up to 8)")
pd.DataFrame(dropped)

In [ ]:
# B. As-of query: risks active on a given date (start <= asof AND (no end OR end > asof))
ASOF = "2024-06-30"
asof_counts = run_cypher("""
    MATCH (c:Company)-[d:DISCLOSES_RISK]->(rf:RiskFactor)
    WHERE d.start_date <= date($asof) AND (d.end_date IS NULL OR d.end_date > date($asof))
    RETURN c.name AS company, count(DISTINCT rf.lineage_id) AS active_risk_lineages
    ORDER BY active_risk_lineages DESC""", asof=ASOF)
print(f"active risk lineages as of {ASOF}:")
pd.DataFrame(asof_counts)

In [ ]:
# C. Newly introduced risks: lineages whose first disclosure is the company's LATEST annual
new_risks = run_cypher("""
    MATCH (c:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)
    WITH c, rf, d MATCH (c)-[:FILED]->(f:Filing) WHERE f.form IN ['10-K','20-F']
    WITH c, rf, d, max(f.filing_date) AS latest
    WHERE rf.first_seen = latest AND rf.category IN ['Export Controls', 'Geopolitical', 'Supply Chain']
    RETURN c.name AS company, rf.category AS category, left(rf.summary, 80) AS new_risk
    ORDER BY company LIMIT 15""")
print("supply-chain/geopolitical risks INTRODUCED in the latest annual filings:")
pd.DataFrame(new_risks)

In [ ]:
# --- M5 (temporal) assertion cell ---
multi_annual = [s for s in lineage_stats if s["annuals"] >= 2]
assert len(multi_annual) >= 8, f"expected >=8 companies with 2+ annual filings, got {len(multi_annual)}"
assert len(updates_df) == len(risks), "every risk node must receive a temporal state"
n_deleted = int((updates_df['status'] == 'Deleted').sum())
n_backdated = int((updates_df['first_seen'] < updates_df['last_seen']).sum())
assert n_backdated > 0, "no recurring risk lineages found across years — check clustering threshold"
with driver.session() as s:
    leak = s.run("""MATCH ()-[d:DISCLOSES_RISK {status:'Deleted'}]->() 
                    WHERE d.end_date IS NULL RETURN count(*) AS n""").single()["n"]
assert leak == 0, f"{leak} Deleted edges lack end_date"
driver.close()
print(f"M5 (temporal) COMPLETE — {len(updates_df)} risk states written: "
      f"{n_deleted} deleted, {n_backdated} recurring-lineage members backdated; no state leaks")